# EDSS 대학원 OpenID 후보 추론

## tl;dr

2023–2024 취업통계의 대학원 학교-연도 2,435개를 대상으로, 대학알리미 이름 맥락·EDSS 대학원 종류별 패널·0101 지역/본분교·0104 학과/주야간·연도 연속성을 결합했다. 결과는 **검토용 후보**이며 원본이나 정식 `개방ID`를 수정하지 않는다.

## Context & Methods

- 대상: `개방ID`가 제거된 EDSS 취업통계 2023–2024 학교·학과 집계.
- 후보 우주: 대학정보공시 0402(일반), 0403(전문), 0404(특수) 대학원의 정상 OpenID.
- 차단 조건: 대학원 종류, 연도, 0101 시도, 본분교.
- 신호: 취업 학과 집합, 대학알리미 2025 학과 및 주야간, 0104 공시 학과, 2023↔2024 연속성.
- 2024 처리: 종류별 ID 패널이 2023년에 끝나므로, 2023 서명을 2024 0101에 존재하는 ID에만 이월하고 신뢰도를 한 단계 낮춘다.
- 보수 장치: 연도 간 후보 충돌과 한 ID가 같은 해 여러 학교를 선택하는 역방향 충돌을 제외한다.
- 임계값: 2022년 정상 ID를 가린 백테스트에서 정밀도 99% 이상을 만족하는 조합 중 커버리지가 가장 높은 값을 사용한다.
- 가정: 학과명 정규화 후 집합 유사성은 기관 식별의 보조 신호다. 학교 개편·학과 개폐·보고 단위 변경 때문에 공식 교차표로 간주하지 않는다.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data/metadata/edss_graduate_open_id_inference.json").exists():
    ROOT = Path("/Users/joocheol/Documents/ChatGPT/EDSS")
report_path = ROOT / "data/metadata/edss_graduate_open_id_inference.json"
candidate_path = ROOT / "data/metadata/edss_graduate_open_id_candidates.csv"
report = json.loads(report_path.read_text(encoding="utf-8"))
candidates = pd.read_csv(candidate_path, dtype=str, keep_default_na=False)
{"rows": len(candidates), "columns": len(candidates.columns), "report_status": report["status"]}

{'rows': 2435, 'columns': 32, 'report_status': 'review_required'}

## Data

아래는 행 수·키 유일성·핵심 결측과 입력 체크섬 영수증이다.

In [2]:
pd.DataFrame({
    "metric": ["rows", "unique_identity_keys", "duplicate_identity_keys", "matched_name_context", "canonical_id_imputed"],
    "value": [report["row_count"], report["unique_identity_key_count"], report["duplicate_identity_key_count"], report["name_context_matched_count"], report["canonical_open_id_imputed_row_count"]],
})

,metric,value
0,rows,2435
1,unique_identity_keys,2435
2,duplicate_identity_keys,0
3,matched_name_context,2286
4,canonical_id_imputed,0


In [3]:
pd.DataFrame(report["inputs"]).T[[c for c in ["path", "bytes", "sha256", "sha256_source"] if c in pd.DataFrame(report["inputs"]).T.columns]]

,path,bytes,sha256,sha256_source
name_candidates,/Users/joocheol/Documents/GitHub/edss/data/met...,NaN,3afa1c2fce0fa1d0fed39f244d71e7a7af94186cabcf4b...,NaN
identity_candidates,/Users/joocheol/Documents/GitHub/edss/data/met...,NaN,14d87910741df657b21922c095c0ba46981ef5ab408ec1...,NaN
employment_aggregate,/Users/joocheol/Documents/GitHub/edss/data/pro...,NaN,ee68db81e54ac5b0bcb01362d2f1067af33d066f53a629...,NaN
academyinfo_graduate_major,/Users/joocheol/Documents/GitHub/edss/data/raw...,NaN,a39e0c91585f4c017e31a86beab6eb4dafe1e27dd35692...,NaN
school_year_bridge,/Users/joocheol/Documents/GitHub/edss/data/met...,NaN,edfbdbf2cc9c20da00f47d059d668eb8fa4469fae79c6c...,NaN
duckdb_build,/Users/joocheol/Documents/GitHub/edss/data/met...,NaN,0f5a82fab70247906548149fe8eafdbea45775c7e20fc0...,NaN
duckdb,/Users/joocheol/Documents/GitHub/edss/data/pro...,16044535808,5a019a6844a8ce4960a4e736fccb497cf278b55b36e09c...,/Users/joocheol/Documents/GitHub/edss/data/met...


## Results

`high`는 같은 해 학과 3개 이상 완전일치, `strong`은 작은 완전일치·백테스트 임계값 통과·검증된 전년도 서명 이월·연도 앵커다. 모두 후보 라벨이며 공식 ID 확정이 아니다.

In [4]:
pd.DataFrame({
    "confidence_tier": list(report["confidence_tier_counts"]),
    "school_year_rows": list(report["confidence_tier_counts"].values()),
}).sort_values("school_year_rows", ascending=False)

,confidence_tier,school_year_rows
1,strong,1270
2,unresolved,630
0,high,535


In [5]:
pd.crosstab(candidates["_panel_year"], candidates["confidence_tier"], margins=True)

confidence_tier,high,strong,unresolved,All
_panel_year,,,,
2023,535,377,297,1209
2024,0,893,333,1226
All,535,1270,630,2435


In [6]:
pd.DataFrame({
    "resolution_status": list(report["resolution_status_counts"]),
    "school_year_rows": list(report["resolution_status_counts"].values()),
}).sort_values("school_year_rows", ascending=False).reset_index(drop=True)

,resolution_status,school_year_rows
0,candidate_high_exact_context,535
1,candidate_strong_exact_prior_year_signature,426
2,candidate_strong_calibrated_multisource,407
3,candidate_strong_exact_small_signature,347
4,unresolved_no_department_overlap,211
5,excluded_reverse_candidate_collision,180
6,unresolved_name_context,149
7,candidate_strong_cross_year_anchor,90
8,unresolved_below_calibrated_threshold,73
9,unresolved_ambiguous_exact_signature,11


### Masked-ID backtest

In [7]:
pd.Series(report["masked_id_backtest"], name="value").to_frame()

,value
year,2022
evaluation_profile_count,1195
selected_profile_count,1079
precision,0.994439
coverage,0.902929
score_threshold,0.6
margin_threshold,0.03
minimum_intersection,1
target_precision,0.99
caveat,Masked-ID backtest uses 0104 department profil...


### Collision and temporal QA

In [8]:
pd.DataFrame({
    "check": ["cross_year_conflict_excluded", "reverse_collision_excluded", "duplicate_identity_key", "canonical_open_id_imputed"],
    "row_count": [report["cross_year_conflict_excluded_row_count"], report["reverse_collision_excluded_row_count"], report["duplicate_identity_key_count"], report["canonical_open_id_imputed_row_count"]],
})

,check,row_count
0,cross_year_conflict_excluded,6
1,reverse_collision_excluded,180
2,duplicate_identity_key,0
3,canonical_open_id_imputed,0


In [9]:
review = candidates.loc[candidates["candidate_open_id"].ne(""), [
    "_panel_year", "edss_school_name", "edss_school_kind", "candidate_open_id",
    "candidate_signature_year", "confidence_tier", "candidate_method", "score",
    "department_jaccard", "academyinfo_department_jaccard", "cross_year_consistent"
]]
review.head(20)

,_panel_year,edss_school_name,edss_school_kind,candidate_open_id,candidate_signature_year,confidence_tier,candidate_method,score,department_jaccard,academyinfo_department_jaccard,cross_year_consistent
0,2023,가야대학교일반대학원,일반대학원,7756780624,2023,strong,unique_exact_small_department_set_with_context,0.866667,1.000000,0.333333,true
1,2023,가천대학교 일반대학원,일반대학원,8810827076,2023,high,unique_exact_kind_year_region_branch_departmen...,0.894118,1.000000,0.470588,true
2,2023,가톨릭관동대학교 일반대학원,일반대학원,4035072386,2023,high,unique_exact_kind_year_region_branch_departmen...,0.885714,1.000000,0.428571,true
5,2023,가톨릭대학교대학원,일반대학원,3736957192,2023,strong,calibrated_multisource_rank_kind_year_region_b...,0.765162,0.785714,0.415094,true
6,2023,감리교신학대학교대학원,일반대학원,1971639871,2023,strong,unique_exact_small_department_set_with_context,0.900000,1.000000,0.500000,true
7,2023,강남대학교대학원,일반대학원,3989555835,2023,high,unique_exact_kind_year_region_branch_departmen...,0.900000,1.000000,0.500000,true
10,2023,강서대학교 일반대학원,일반대학원,8083013415,2023,high,unique_exact_kind_year_region_branch_departmen...,0.876786,1.000000,0.428571,true
11,2023,강원대학교일반대학원,일반대학원,2172015368,2023,high,unique_exact_kind_year_region_branch_departmen...,0.892147,1.000000,0.474886,true
13,2023,건국대학교일반대학원,일반대학원,8120602428,2023,strong,calibrated_multisource_rank_kind_year_region_b...,0.780294,0.810811,0.421739,true
15,2023,경기대학교 일반대학원,일반대학원,7891201674,2023,high,unique_exact_kind_year_region_branch_departmen...,0.896610,1.000000,0.483051,not_tested_single_year


## Takeaways

- 2,435개 학교-연도 중 1,805개에 역방향 충돌을 통과한 검토 후보가 남았다: high 535개, strong 1,270개.
- 630개는 미해결이다. 이 중 이름 맥락 미연결, 학과 겹침 없음, 역방향 충돌 등이 명시적으로 구분된다.
- 2022 마스킹 백테스트는 선택된 프록시 표본에서 정밀도 약 99.44%, 커버리지 약 90.29%다. 다만 0104를 취업통계의 대리 입력으로 쓴 결과이므로 실제 정밀도의 보증은 아니다.
- 안전한 다음 단계는 high 후보부터 학교 공식 대학원 페이지/대학알리미 추가 연도와 대조해 승인 목록을 별도로 만드는 것이다. `candidate_open_id`를 원본 또는 정식 `개방ID`에 자동 복사하면 안 된다.